#### Gaussian Mixture Model

In [20]:
from sklearn.mixture import GaussianMixture
from sklearn.metrics import silhouette_score, davies_bouldin_score, adjusted_rand_score, normalized_mutual_info_score
from sklearn.manifold import TSNE
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd

In [21]:
data = pd.read_csv("../../../data/preprocessed/preprocessed_dataset.csv")
y = data["expression"].astype("category").cat.codes

datasets = {
    "Scaled dataset": "../../../data/reduced/X_scaled.npy",
    "PCA dataset": "../../../data/reduced/X_pca.npy"
}

In [22]:
def select_gmm_components(X, dataset_name, max_components=10, covariance_type="diag"):
    bics = []
    components_range = range(2, max_components + 1)

    for n in components_range:
        gmm = GaussianMixture(
            n_components=n,
            covariance_type=covariance_type,
            random_state=42
        )
        gmm.fit(X)
        bics.append(gmm.bic(X))

    best_n = components_range[np.argmin(bics)]

    plt.figure(figsize=(7, 5))
    plt.plot(list(components_range), bics, marker='o')
    plt.xlabel("Number of components")
    plt.ylabel("BIC")
    plt.title(f"GMM BIC - {dataset_name}")
    plt.grid(True)
    plt.tight_layout()
    plt.savefig(f"../../../results/gmm/gmm_bic_{dataset_name}.png", dpi=200)
    plt.close()
    # plt.show()

    return best_n

##### Zbog velikog broja dimenzija korišćen je diag tip kovarijacione matrice, jer full značajno povećava računarsku složenost algoritma.

In [23]:
results = []

for dataset_name, X_data in datasets.items():
    X = np.load(X_data)

    n_components = select_gmm_components(
        X,
        dataset_name,
        max_components=10,
        covariance_type="diag"
    )
    print(f"Optimal number of components for {dataset_name} according to BIC: {n_components}")

    gmm = GaussianMixture(
        n_components=n_components,
        covariance_type="diag",
        random_state=42
    )
    labels = gmm.fit_predict(X)

    row = {
        "Dataset": dataset_name,
        "Algorithm": "GMM",
        "K": n_components,
        "Silhouette": silhouette_score(X, labels, sample_size=3000),
        "Davies_Bouldin": davies_bouldin_score(X, labels),
        "ARI": adjusted_rand_score(y, labels),
        "NMI": normalized_mutual_info_score(y, labels)
    }
    results.append(row)

    if dataset_name == "PCA dataset":
        tsne = TSNE(n_components=2, perplexity=30, random_state=42)
        X_tsne = tsne.fit_transform(X)

        plt.figure(figsize=(6, 5))
        scatter = plt.scatter(X_tsne[:, 0], X_tsne[:, 1], c=labels, s=8, cmap="tab10")
        plt.title(f"GMM clustering on {dataset_name}")
        plt.xlabel("t-SNE 1")
        plt.ylabel("t-SNE 2")
        plt.colorbar(scatter, label="Cluster")
        plt.tight_layout()
        plt.savefig(f"../../../results/gmm/gmmTSNE_{dataset_name}.png", dpi=200)
        plt.close()
        # plt.show()

        plt.figure(figsize=(6,5))
        plt.scatter(X_tsne[:,0], X_tsne[:,1], c=y, s=5, cmap="tab10")
        plt.title(f"True classes TSNE ({dataset_name})")
        plt.tight_layout()
        plt.savefig(f"../../../results/gmm/trueTSNE_{dataset_name}.png", dpi=200)
        plt.close()

Optimal number of components for Scaled dataset according to BIC: 10
Optimal number of components for PCA dataset according to BIC: 10


##### Gaussian Mixture Model (GMM) je probabilistički model klasterovanja koji pretpostavlja da podaci potiču iz mešavine više Gaussovih raspodela. Za razliku od KMeans algoritma, koji svakom uzorku dodeljuje tačno jedan klaster, GMM omogućava meku pripadnost, odnosno procenu verovatnoće da uzorak pripada svakom klasteru. Broj komponenti je određen pomoću BIC kriterijuma.

In [24]:
results_df = pd.DataFrame(results)
results_df = results_df.round({
    "Silhouette": 4,
    "Davies_Bouldin": 4,
    "ARI": 4,
    "NMI": 4
})
results_df.to_csv("../../../results/gmm/gmm_results_summary.csv", index=False)
print(results_df)

          Dataset Algorithm   K  Silhouette  Davies_Bouldin     ARI     NMI
0  Scaled dataset       GMM  10      0.1030          2.6087  0.1102  0.2311
1     PCA dataset       GMM  10     -0.0898          5.0807  0.0498  0.1204


##### Eksperimenti pokazuju da KMeans daje najstabilnije rezultate na analiziranom skupu podataka. Agglomerative clustering postiže slične performanse, ali sa nešto slabijom separacijom klastera. Gaussian Mixture Model pokazuje značajno slabije rezultate, posebno na PCA datasetu, što ukazuje da podaci verovatno ne prate pretpostavku mešavine Gaussovih raspodela.

#### Gaussian Method with k = 4 fixed components

In [25]:
results = []

for dataset_name, X_data in datasets.items():
    X = np.load(X_data)

    n_components = 4
    # print(f"Optimal number of components for {dataset_name} according to BIC: {n_components}")

    gmm = GaussianMixture(
        n_components=n_components,
        covariance_type="diag",
        random_state=42
    )
    labels = gmm.fit_predict(X)

    row = {
        "Dataset": dataset_name,
        "Algorithm": "GMM",
        "K": n_components,
        "Silhouette": silhouette_score(X, labels, sample_size=3000),
        "Davies_Bouldin": davies_bouldin_score(X, labels),
        "ARI": adjusted_rand_score(y, labels),
        "NMI": normalized_mutual_info_score(y, labels)
    }
    results.append(row)

    if dataset_name == "PCA dataset":
        tsne = TSNE(n_components=2, perplexity=30, random_state=42)
        X_tsne = tsne.fit_transform(X)

        plt.figure(figsize=(6, 5))
        scatter = plt.scatter(X_tsne[:, 0], X_tsne[:, 1], c=labels, s=8, cmap=plt.cm.get_cmap("tab10", 4))
        plt.title(f"GMM clustering on {dataset_name} (K={n_components})")
        plt.xlabel("t-SNE 1")
        plt.ylabel("t-SNE 2")
        plt.colorbar(scatter, label="Cluster")
        plt.tight_layout()
        plt.savefig(f"../../../results/gmm/gmmTSNE_{dataset_name}_fixedK.png", dpi=200)
        plt.close()
        # plt.show()

results_df = pd.DataFrame(results)
results_df = results_df.round({
    "Silhouette": 4,
    "Davies_Bouldin": 4,
    "ARI": 4,
    "NMI": 4
})

results_df.to_csv("../../../results/gmm/gmm_fixed_k_summary.csv", index=False)
print(results_df)

          Dataset Algorithm  K  Silhouette  Davies_Bouldin     ARI     NMI
0  Scaled dataset       GMM  4      0.3192          2.3329  0.0345  0.0779
1     PCA dataset       GMM  4      0.0223          3.9672  0.0518  0.0913


##### Gaussian Mixture Model je primenjen sa četiri komponente kako bi poređenje sa ostalim algoritmima bilo metodološki konzistentno. Rezultati pokazuju da GMM daje solidne performanse na skaliranom datasetu (Silhouette ≈ 0.32), ali značajno slabije rezultate na PCA datasetu. U poređenju sa KMeans i aglomerativnim klasterovanjem, GMM pokazuje veću tendenciju preklapanja klastera.